# Northwind Traders — Single Master Dataset

This notebook combines all 7 source CSVs into **one clean, analysis-ready dataset**.

| Source Table | Rows | Join Key |
|---|---|---|
| order_details | 2155 | orderID, productID |
| orders | 830 | orderID |
| products | 77 | productID |
| categories | 8 | categoryID |
| customers | 91 | customerID |
| employees | 9 | employeeID |
| shippers | 3 | shipperID |

**Final Output:** `northwind_master.csv`

## Step 1 : Import Libraries

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries imported.')

Libraries imported.


## Step 2 : Load All Source CSVs

In [2]:
order_details = pd.read_csv('order_details.csv')
orders        = pd.read_csv('orders.csv')
products      = pd.read_csv('products.csv',   encoding='latin1')
categories    = pd.read_csv('categories.csv')
customers     = pd.read_csv('customers.csv',  encoding='latin1')
employees     = pd.read_csv('employees.csv')
shippers      = pd.read_csv('shippers.csv')

print('All CSVs loaded successfully.')
print()
print(f'  order_details : {order_details.shape}')
print(f'  orders        : {orders.shape}')
print(f'  products      : {products.shape}')
print(f'  categories    : {categories.shape}')
print(f'  customers     : {customers.shape}')
print(f'  employees     : {employees.shape}')
print(f'  shippers      : {shippers.shape}')

All CSVs loaded successfully.

  order_details : (2155, 5)
  orders        : (830, 8)
  products      : (77, 6)
  categories    : (8, 3)
  customers     : (91, 6)
  employees     : (9, 6)
  shippers      : (3, 2)


## Step 3 : Merge All Tables Into One Dataset

Join order: `order_details` → `products` → `categories` → `orders` → `customers` → `employees` → `shippers`

In [3]:
# --- Merge 1: order_details + products ---
df = order_details.merge(
    products[['productID', 'productName', 'quantityPerUnit', 'unitPrice', 'categoryID', 'discontinued']]
    .rename(columns={'unitPrice': 'product_unitPrice'}),
    on='productID', how='left'
)

# --- Merge 2: + categories ---
df = df.merge(
    categories.rename(columns={'description': 'category_description'}),
    on='categoryID', how='left'
)

# --- Merge 3: + orders ---
df = df.merge(
    orders,
    on='orderID', how='left'
)

# --- Merge 4: + customers ---
df = df.merge(
    customers.rename(columns={
        'companyName':  'customer_company',
        'contactName':  'customer_contact',
        'contactTitle': 'customer_title',
        'city':         'customer_city',
        'country':      'customer_country'
    }),
    on='customerID', how='left'
)

# --- Merge 5: + employees ---
df = df.merge(
    employees.rename(columns={
        'employeeName': 'employee_name',
        'title':        'employee_title',
        'city':         'employee_city',
        'country':      'employee_country',
        'reportsTo':    'reports_to'
    }),
    on='employeeID', how='left'
)

# --- Merge 6: + shippers ---
df = df.merge(
    shippers.rename(columns={'companyName': 'shipper_name'}),
    on='shipperID', how='left'
)

print(f'All tables merged. Shape: {df.shape}')

All tables merged. Shape: (2155, 30)

## Step 4 : Add Calculated Columns

In [4]:
# Revenue per line item after discount
df['line_total'] = df['unitPrice'] * df['quantity'] * (1 - df['discount'])

# Parse dates
df['orderDate']    = pd.to_datetime(df['orderDate'])
df['requiredDate'] = pd.to_datetime(df['requiredDate'])
df['shippedDate']  = pd.to_datetime(df['shippedDate'])

# Derived date columns
df['order_year']    = df['orderDate'].dt.year
df['order_month']   = df['orderDate'].dt.month
df['order_quarter'] = df['orderDate'].dt.quarter

# Shipping delay in days
df['ship_delay_days'] = (df['shippedDate'] - df['orderDate']).dt.days

print('Calculated columns added:')
print('  line_total, order_year, order_month, order_quarter, ship_delay_days')

Calculated columns added:
  line_total, order_year, order_month, order_quarter, ship_delay_days


## Step 5 : Clean Column Order

In [5]:
final_columns = [
    # Order info
    'orderID', 'orderDate', 'order_year', 'order_month', 'order_quarter',
    'requiredDate', 'shippedDate', 'ship_delay_days', 'freight',

    # Product info
    'productID', 'productName', 'quantityPerUnit', 'categoryID', 'categoryName', 'category_description',
    'discontinued',

    # Order line detail
    'unitPrice', 'quantity', 'discount', 'line_total',

    # Customer info
    'customerID', 'customer_company', 'customer_contact', 'customer_title',
    'customer_city', 'customer_country',

    # Employee info
    'employeeID', 'employee_name', 'employee_title', 'employee_city', 'employee_country', 'reports_to',

    # Shipper info
    'shipperID', 'shipper_name'
]

df = df[final_columns]
print(f'Final dataset shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')

Final dataset shape: (2155, 34)
Columns (34): ['orderID', 'orderDate', 'order_year', 'order_month', 'order_quarter', 'requiredDate', 'shippedDate', 'ship_delay_days', 'freight', 'productID', 'productName', 'quantityPerUnit', 'categoryID', 'categoryName', 'category_description', 'discontinued', 'unitPrice', 'quantity', 'discount', 'line_total', 'customerID', 'customer_company', 'customer_contact', 'customer_title', 'customer_city', 'customer_country', 'employeeID', 'employee_name', 'employee_title', 'employee_city', 'employee_country', 'reports_to', 'shipperID', 'shipper_name']


## Step 6 : Check for Missing Values

In [6]:
missing = df.isnull().sum()
missing_df = missing[missing > 0].reset_index()
missing_df.columns = ['Column', 'Missing Count']
missing_df['Missing %'] = (missing_df['Missing Count'] / len(df) * 100).round(2)

if missing_df.empty:
    print('No missing values found.')
else:
    print('Columns with missing values:')
    display(missing_df)

Columns with missing values:


,Column,Missing Count,Missing %
0,shippedDate,73,3.39
1,ship_delay_days,73,3.39
2,reports_to,241,11.18


## Step 7 : Data Types Overview

In [7]:
dtype_df = pd.DataFrame({
    'Column':   df.columns,
    'DataType': df.dtypes.values,
    'Non-Null': df.notnull().sum().values,
    'Null':     df.isnull().sum().values
})
display(dtype_df)

,Column,DataType,Non-Null,Null
0,orderID,int64,2155,0
1,orderDate,datetime64[ns],2155,0
2,order_year,int32,2155,0
3,order_month,int32,2155,0
4,order_quarter,int32,2155,0
...,...,...,...,...
29,employee_city,object,2155,0
30,employee_country,object,2155,0
31,reports_to,float64,1914,241
32,shipperID,int64,2155,0


## Step 8 : Preview the Final Dataset

In [8]:
print(f'Total rows : {len(df)}')
print(f'Total cols : {len(df.columns)}')
print()
df.head(10)

Total rows : 2155
Total cols : 34



,orderID,orderDate,order_year,order_month,order_quarter,requiredDate,shippedDate,ship_delay_days,freight,productID,productName,quantityPerUnit,categoryID,categoryName,category_description,discontinued,unitPrice,quantity,discount,line_total,customerID,customer_company,customer_contact,customer_title,customer_city,customer_country,employeeID,employee_name,employee_title,employee_city,employee_country,reports_to,shipperID,shipper_name
0,10248,2013-07-04,2013,7,3,2013-08-01,2013-07-16,12.00,32.38,11,Queso Cabrales,1 kg pkg.,4,Dairy Products,Cheeses,0,14.00,12,0.00,168.00,VINET,Vins et alcools Chevalier,Paul Henriot,Accounting Manager,Reims,France,5,Steven Buchanan,Sales Manager,London,UK,2.00,3,Federal Shipping
1,10248,2013-07-04,2013,7,3,2013-08-01,2013-07-16,12.00,32.38,42,Singaporean Hokkien Fried Mee,32 - 1 kg pkgs.,5,Grains & Cereals,"Breads, crackers, pasta, and cereal",1,9.80,10,0.00,98.00,VINET,Vins et alcools Chevalier,Paul Henriot,Accounting Manager,Reims,France,5,Steven Buchanan,Sales Manager,London,UK,2.00,3,Federal Shipping
2,10248,2013-07-04,2013,7,3,2013-08-01,2013-07-16,12.00,32.38,72,Mozzarella di Giovanni,24 - 200 g pkgs.,4,Dairy Products,Cheeses,0,34.80,5,0.00,174.00,VINET,Vins et alcools Chevalier,Paul Henriot,Accounting Manager,Reims,France,5,Steven Buchanan,Sales Manager,London,UK,2.00,3,Federal Shipping
3,10249,2013-07-05,2013,7,3,2013-08-16,2013-07-10,5.00,11.61,14,Tofu,40 - 100 g pkgs.,7,Produce,Dried fruit and bean curd,0,18.60,9,0.00,167.40,TOMSP,Toms Spezialitäten,Karin Josephs,Marketing Manager,Münster,Germany,6,Michael Suyama,Sales Representative,London,UK,5.00,1,Speedy Express
4,10249,2013-07-05,2013,7,3,2013-08-16,2013-07-10,5.00,11.61,51,Manjimup Dried Apples,50 - 300 g pkgs.,7,Produce,Dried fruit and bean curd,0,42.40,40,0.00,1696.00,TOMSP,Toms Spezialitäten,Karin Josephs,Marketing Manager,Münster,Germany,6,Michael Suyama,Sales Representative,London,UK,5.00,1,Speedy Express
5,10250,2013-07-08,2013,7,3,2013-08-05,2013-07-12,4.00,65.83,41,Jack's New England Clam Chowder,12 - 12 oz cans,8,Seafood,Seaweed and fish,0,7.70,10,0.00,77.00,HANAR,Hanari Carnes,Mario Pontes,Accounting Manager,Rio de Janeiro,Brazil,4,Margaret Peacock,Sales Representative,New York,USA,8.00,2,United Package
6,10250,2013-07-08,2013,7,3,2013-08-05,2013-07-12,4.00,65.83,51,Manjimup Dried Apples,50 - 300 g pkgs.,7,Produce,Dried fruit and bean curd,0,42.40,35,0.15,1261.40,HANAR,Hanari Carnes,Mario Pontes,Accounting Manager,Rio de Janeiro,Brazil,4,Margaret Peacock,Sales Representative,New York,USA,8.00,2,United Package
7,10250,2013-07-08,2013,7,3,2013-08-05,2013-07-12,4.00,65.83,65,Louisiana Fiery Hot Pepper Sauce,32 - 8 oz bottles,2,Condiments,"Sweet and savory sauces, relishes, spreads, an...",0,16.80,15,0.15,214.20,HANAR,Hanari Carnes,Mario Pontes,Accounting Manager,Rio de Janeiro,Brazil,4,Margaret Peacock,Sales Representative,New York,USA,8.00,2,United Package
8,10251,2013-07-08,2013,7,3,2013-08-05,2013-07-15,7.00,41.34,22,Gustaf's Knackebröd,24 - 500 g pkgs.,5,Grains & Cereals,"Breads, crackers, pasta, and cereal",0,16.80,6,0.05,95.76,VICTE,Victuailles en stock,Mary Saveley,Sales Agent,Lyon,France,3,Janet Leverling,Sales Representative,New York,USA,8.00,1,Speedy Express
9,10251,2013-07-08,2013,7,3,2013-08-05,2013-07-15,7.00,41.34,57,Ravioli Angelo,24 - 250 g pkgs.,5,Grains & Cereals,"Breads, crackers, pasta, and cereal",0,15.60,15,0.05,222.30,VICTE,Victuailles en stock,Mary Saveley,Sales Agent,Lyon,France,3,Janet Leverling,Sales Representative,New York,USA,8.00,1,Speedy Express


In [9]:
df.tail(5)

,orderID,orderDate,order_year,order_month,order_quarter,requiredDate,shippedDate,ship_delay_days,freight,productID,productName,quantityPerUnit,categoryID,categoryName,category_description,discontinued,unitPrice,quantity,discount,line_total,customerID,customer_company,customer_contact,customer_title,customer_city,customer_country,employeeID,employee_name,employee_title,employee_city,employee_country,reports_to,shipperID,shipper_name
2150,11077,2015-05-06,2015,5,2,2015-06-03,NaT,NaN,8.53,64,Wimmers gute Semmelknödel,20 bags x 4 pieces,5,Grains & Cereals,"Breads, crackers, pasta, and cereal",0,33.25,2,0.03,64.50,RATTC,Rattlesnake Canyon Grocery,Paula Wilson,Assistant Sales Representative,Albuquerque,USA,1,Nancy Davolio,Sales Representative,New York,USA,8.00,2,United Package
2151,11077,2015-05-06,2015,5,2,2015-06-03,NaT,NaN,8.53,66,Louisiana Hot Spiced Okra,24 - 8 oz jars,2,Condiments,"Sweet and savory sauces, relishes, spreads, an...",0,17.00,1,0.00,17.00,RATTC,Rattlesnake Canyon Grocery,Paula Wilson,Assistant Sales Representative,Albuquerque,USA,1,Nancy Davolio,Sales Representative,New York,USA,8.00,2,United Package
2152,11077,2015-05-06,2015,5,2,2015-06-03,NaT,NaN,8.53,73,Röd Kaviar,24 - 150 g jars,8,Seafood,Seaweed and fish,0,15.00,2,0.01,29.70,RATTC,Rattlesnake Canyon Grocery,Paula Wilson,Assistant Sales Representative,Albuquerque,USA,1,Nancy Davolio,Sales Representative,New York,USA,8.00,2,United Package
2153,11077,2015-05-06,2015,5,2,2015-06-03,NaT,NaN,8.53,75,Rhönbräu Klosterbier,24 - 0.5 l bottles,1,Beverages,"Soft drinks, coffees, teas, beers, and ales",0,7.75,4,0.00,31.00,RATTC,Rattlesnake Canyon Grocery,Paula Wilson,Assistant Sales Representative,Albuquerque,USA,1,Nancy Davolio,Sales Representative,New York,USA,8.00,2,United Package
2154,11077,2015-05-06,2015,5,2,2015-06-03,NaT,NaN,8.53,77,Original Frankfurter Grüne Soße,12 boxes,2,Condiments,"Sweet and savory sauces, relishes, spreads, an...",0,13.00,2,0.00,26.00,RATTC,Rattlesnake Canyon Grocery,Paula Wilson,Assistant Sales Representative,Albuquerque,USA,1,Nancy Davolio,Sales Representative,New York,USA,8.00,2,United Package


In [10]:
df.describe(include='all')

,orderID,orderDate,order_year,order_month,order_quarter,requiredDate,shippedDate,ship_delay_days,freight,productID,productName,quantityPerUnit,categoryID,categoryName,category_description,discontinued,unitPrice,quantity,discount,line_total,customerID,customer_company,customer_contact,customer_title,customer_city,customer_country,employeeID,employee_name,employee_title,employee_city,employee_country,reports_to,shipperID,shipper_name
count,2155.00,2155,2155.00,2155.00,2155.00,2155,2082,2082.00,2155.00,2155.00,2155,2155,2155.00,2155,2155,2155.00,2155.00,2155.00,2155.00,2155.00,2155,2155,2155,2155,2155,2155,2155.00,2155,2155,2155,2155,1914.00,2155.00,2155
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77,70,NaN,8,8,NaN,NaN,NaN,NaN,NaN,89,89,89,12,69,21,NaN,9,3,2,2,NaN,NaN,3
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Raclette Courdavault,24 - 12 oz bottles,NaN,Beverages,"Soft drinks, coffees, teas, beers, and ales",NaN,NaN,NaN,NaN,NaN,SAVEA,Save-a-lot Markets,Jose Pavarotti,Sales Representative,Boise,USA,NaN,Margaret Peacock,Sales Representative,New York,USA,NaN,NaN,United Package
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54,109,NaN,404,404,NaN,NaN,NaN,NaN,NaN,116,116,116,414,116,352,NaN,420,1537,1587,1587,NaN,NaN,864
mean,10659.38,2014-08-05 06:12:11.693735680,2014.13,6.06,2.37,2014-09-02 02:27:40.510440704,2014-08-04 04:05:31.988472576,8.35,96.20,40.79,NaN,NaN,4.14,NaN,NaN,0.11,26.22,23.81,0.06,587.37,NaN,NaN,NaN,NaN,NaN,NaN,4.33,NaN,NaN,NaN,NaN,6.11,2.00,NaN
min,10248.00,2013-07-04 00:00:00,2013.00,1.00,1.00,2013-07-24 00:00:00,2013-07-10 00:00:00,1.00,0.02,1.00,NaN,NaN,1.00,NaN,NaN,0.00,2.00,1.00,0.00,4.80,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,NaN,NaN,2.00,1.00,NaN
25%,10451.00,2014-02-19 00:00:00,2014.00,3.00,1.00,2014-03-18 12:00:00,2014-02-24 00:00:00,4.00,19.61,22.00,NaN,NaN,2.00,NaN,NaN,0.00,12.00,10.00,0.00,147.00,NaN,NaN,NaN,NaN,NaN,NaN,2.00,NaN,NaN,NaN,NaN,5.00,1.00,NaN
50%,10657.00,2014-09-04 00:00:00,2014.00,5.00,2.00,2014-10-02 00:00:00,2014-09-01 00:00:00,7.00,53.80,41.00,NaN,NaN,4.00,NaN,NaN,0.00,18.40,20.00,0.00,337.75,NaN,NaN,NaN,NaN,NaN,NaN,4.00,NaN,NaN,NaN,NaN,8.00,2.00,NaN
75%,10862.50,2015-01-31 12:00:00,2015.00,9.00,3.00,2015-03-03 00:00:00,2015-01-30 00:00:00,9.00,120.92,60.00,NaN,NaN,6.00,NaN,NaN,0.00,32.00,30.00,0.10,656.00,NaN,NaN,NaN,NaN,NaN,NaN,7.00,NaN,NaN,NaN,NaN,8.00,3.00,NaN
max,11077.00,2015-05-06 00:00:00,2015.00,12.00,4.00,2015-06-11 00:00:00,2015-05-06 00:00:00,37.00,1007.64,77.00,NaN,NaN,8.00,NaN,NaN,1.00,263.50,130.00,0.25,15810.00,NaN,NaN,NaN,NaN,NaN,NaN,9.00,NaN,NaN,NaN,NaN,8.00,3.00,NaN


## Step 9 : Quick Summary Stats

In [11]:
print('=== DATASET SUMMARY ===')
print(f'Total Orders     : {df["orderID"].nunique():,}')
print(f'Total Products   : {df["productID"].nunique():,}')
print(f'Total Customers  : {df["customerID"].nunique():,}')
print(f'Total Employees  : {df["employeeID"].nunique():,}')
print(f'Total Categories : {df["categoryID"].nunique():,}')
print(f'Total Shippers   : {df["shipperID"].nunique():,}')
print(f'Date Range       : {df["orderDate"].min().date()} to {df["orderDate"].max().date()}')
print(f'Total Revenue    : ${df["line_total"].sum():,.2f}')
print(f'Avg Order Value  : ${df.groupby("orderID")["line_total"].sum().mean():,.2f}')

=== DATASET SUMMARY ===
Total Orders     : 830
Total Products   : 77
Total Customers  : 89
Total Employees  : 9
Total Categories : 8
Total Shippers   : 3
Date Range       : 2013-07-04 to 2015-05-06
Total Revenue    : $1,265,793.04
Avg Order Value  : $1,525.05


## Step 10 : Export — One Final CSV Dataset

In [12]:
output_file = 'northwind_master.csv'
df.to_csv(output_file, index=False)

print(f'Saved: {output_file}')
print(f'Rows  : {len(df):,}')
print(f'Cols  : {len(df.columns)}')
print(f'Columns: {list(df.columns)}')

Saved: northwind_master.csv
Rows  : 2,155
Cols  : 34
Columns: ['orderID', 'orderDate', 'order_year', 'order_month', 'order_quarter', 'requiredDate', 'shippedDate', 'ship_delay_days', 'freight', 'productID', 'productName', 'quantityPerUnit', 'categoryID', 'categoryName', 'category_description', 'discontinued', 'unitPrice', 'quantity', 'discount', 'line_total', 'customerID', 'customer_company', 'customer_contact', 'customer_title', 'customer_city', 'customer_country', 'employeeID', 'employee_name', 'employee_title', 'employee_city', 'employee_country', 'reports_to', 'shipperID', 'shipper_name']
